# Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv("./data/download/질병.csv")

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")
df.head(2)

(전체 데이터 수, 전체 컬럼 수): (130, 5)


,video_url,video_id,title,channel,caption
0,https://www.youtube.com/watch?v=8GwNlF44ntY&li...,8GwNlF44ntY,#164 식은땀! 이 정도는 알아두셔야 합니다. : 하정훈의 육아이야기,하정훈의 삐뽀삐뽀 119 소아과 (삐뽀삐뽀 119 소아과),[음악] 안녕하세요 소아청소년과 전문의 가져옵니다 이제는 많이 줄긴 했지만 아직도 ...
1,https://www.youtube.com/watch?v=8GwNlF44ntY,8GwNlF44ntY,#164 식은땀! 이 정도는 알아두셔야 합니다. : 하정훈의 육아이야기,하정훈의 삐뽀삐뽀 119 소아과 (삐뽀삐뽀 119 소아과),[음악] 안녕하세요 소아청소년과 전문의 가져옵니다 이제는 많이 줄긴 했지만 아직도 ...


# Drop duplicates

In [2]:
df.drop_duplicates(inplace=True, subset=['video_id'])

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")

(전체 데이터 수, 전체 컬럼 수): (129, 5)


# Cleaning Caption

In [3]:
def cleaning_caption(data:str) -> str:
  return data.replace(" 으 ", "").replace(" 아 ", "")\
    .replace("[음악]", "").replace("[박수]", "").strip()


In [4]:
df['caption'] = df['caption'].map(cleaning_caption)


In [5]:
df.loc[0,['caption']].values

array(['안녕하세요 소아청소년과 전문의 가져옵니다 이제는 많이 줄긴 했지만 아직도 우리 아기가 시간 다 나았는데 어디 약한거 아닌가 라는 이야기 하시는 분 종종 있습니다 감기 약이라도 먹고 있으면 약이 너무 센 것은 아닌가 라고 물어보기도 합니다 이번에는 많은 부모들이 고민하고 있는 식은 땀 이라는 것에 대해서 말씀을 드리겠습니다 사람들은 원래 몸에 담이 나게 되어 있죠 사람 몸에는 200만 개가 되는 땅 팀이 있어 땀을 흘리게 되는데 땀 이란 우리 몸 안의 노폐물을 배출하고 체온을 조절하는 역할을 하게 됩니다 적당히 땀을 흘리는 것은 정상입니다 사람마다 차이가 크지만 아이들은 어른의 피에서 남을 드 많이 걸립니다 게다가 아이들은 땀을 조절하는 기능이 어른들의 비해서 미숙하기 때문에 땅 세미나는 이 많았 인물이나 성 바닥 발바닥은 평소에도 식사를 하거나 조금 말 힘이 들어도 땀이 송골송골 맺히는 경우가 흔합니다 별다른 문제가 없어도 베개가 땀으로 젖는 경우도 있습니다 물론 감기에 걸리거나 다른 이상이 있는 경우에도 땅 조절하는 기능이 더 떨어져서 남을 과잉생산 하기도 합니다 아이들은 원래 땀이 많은 보입니다 그런데 아이가 땀을 많이 흘리면 몸이 허약하다 고 생각하는 부모들이 많은 것이 현실입니다 아기가 빔 아닌데도 땀을 많이 흘리니 까 몸이 허약하다 고 몸보신을 해줘야 한다고 말하는 부모도 있습니다 땀을 흘린다고 영양제 먹이는 부모도 많은 것이 현실입니다 부모들이 그렇게도 고민고민 하면서 운이 않은 아이들이 시킨 다음에 대해서 소아 청소에 가설은 별로 심각하게 생각하지 않습니다 아이가 땀을 많이 클리 더라도 열이 없고 으로 보기에 다른 이상이 없는 경우에는 대개 체계적인 경우라고 생각합니다 특히 아기가 어릴수록 땀을 많이 흘리게 되고 평소에 땀을 않으니 다가 갑자기 많이 흘리게 되는 경우도 많습니다 아이들 중에서도 특히 네 아이는 은 아니 식은땀을 많이 흘리는 법입니다 그런데 간혹 진짜 문제가 있는 경우도 있습니다 선천성 심장병이 있거나 갑상선 기능 항진

# LLM을 이용한 Caption 클린징 

## LLM

In [6]:
from dotenv import load_dotenv 

load_dotenv()

True

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",
    reasoning_effort="high",        # 논리성 강화
)

## ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
너는 영상 캡션 데이터를 정제하는 전문가다.

아래 문장은 음성 인식 또는 자동 생성된 캡션이라
깨진 글자, 반복 표현, 의미 없는 소리, 불필요한 기호가 포함되어 있다.

작업 목표:
- 의미는 유지한다
- 자연스럽고 올바른 한국어 문장으로 복원한다
- 새 정보를 추가하지 않는다
- 설명하지 말고 결과 문장만 출력한다

반드시 지켜야 할 치환 규칙:
- "삐뽀삐뽀119소아과" → "소아청소년과"
- "하정훈" → "소아청소년과 전문의"

주의:
- 위 치환은 문맥과 상관없이 **항상 적용**
- 다른 고유명사는 임의로 바꾸지 않는다
- 치환 사실을 설명하거나 주석을 달지 않는다

입력 문장:
{caption}

정제된 문장:
"""
)


## Chain

In [9]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

## Cleaning caption

In [10]:
captions = df["caption"].tolist()

print(f"captions: {len(captions)}")

captions: 129


> Batch 처리용 입력 준비

In [13]:
from itertools import compress


# 유효한 문장만 LLM에 전달
valid_mask = [
    isinstance(c, str) and c.strip()
    for c in captions
]

inputs = [
    {"caption": caption}
    for caption in compress(captions, valid_mask)
]

> Batch + 병렬 실행

In [14]:
from tqdm import tqdm

BATCH_SIZE = 50
MAX_CONCURRENCY = 8 # 최대 8개의 요청을 동시에 날림

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

results = []

for chunk in tqdm(list(chunks(inputs, BATCH_SIZE))):
    results.extend(
        chain.batch(
            chunk,
            config={"max_concurrency": MAX_CONCURRENCY}
        )
    )

100%|██████████| 3/3 [42:59<00:00, 859.85s/it] 


> 결과를 원래 DataFrame에 매핑

In [18]:
cleaned_texts = []
result_idx = 0

for is_valid in valid_mask:
    if is_valid:
        cleaned_texts.append(results[result_idx])
        result_idx += 1
    else:
        cleaned_texts.append(None)

df["clean_caption"] = cleaned_texts

In [22]:
df["clean_caption"] = df["clean_caption"].map(lambda x: x.strip())

In [23]:
df[['clean_caption', 'caption']].head()

,clean_caption,caption
0,"안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가...",안녕하세요 소아청소년과 전문의 가져옵니다 이제는 많이 줄긴 했지만 아직도 우리 아기...
2,"안녕하세요, 소아청소년과 전문의입니다. 날이 더운 여름에는 식중독이 많이 발생합니다...",안녕하세요 소아청소년과 전문의 과정 입니다 날이 더운 여름이 되면 집중 독이 많이 ...
3,"안녕하세요, 소아청소년과 전문의입니다. 최근 안산의 한 유치원에서 집단 식중독이 발...",안녕하세요 서 청순과 전문의 가정입니다 최근 안산의 한 유치원에서 식중독 영상이 발...
4,"안녕하세요, 소아청소년과 전문의입니다. 예전보다는 많이 줄었지만 아직도 설사하는 아...",으 안녕하세요 어화 중 성과 전문의 가져옵니다 예전보다는 많이 줄었지만 아직도 설 ...
5,안녕하세요. 소아청소년과 전문의입니다. 아기들이 사타구니 쪽에 혹이 갑자기 생겼다가...,으 안녕하세요 소아정신과 전문의 가져옵니다 아기들 사타구니 사이가 갑자 이 붙는 경...


In [24]:
df[['clean_caption', 'caption']].tail()

,clean_caption,caption
125,"안녕하세요, 소아청소년과 전문의 하지원입니다. 요즘 코로나에 걸린 아이들이 정말 많...",안녕하세요 서청소년과 전문의 하지원입니다 요즘 코로나 걸린 아이들 정말 많습니다 코...
126,안녕하세요 소아청소년과 전문의입니다. 오늘은 아이가 코로나에 걸렸을 때와 아이를 키...,안녕하세요 소아청소년과 전문의 하정훈입니다 오늘은 아이가 코로나에 걸렸을 때와 아이...
127,"안녕하세요, 소아청소년과 전문의입니다. 최근 오미크론 확산으로 코로나 환자가 급증하...",안녕하세요 소아청소년과 전문의 하정훈입니다 최근 오미크론의 확산으로 코로나 환자가 ...
128,"안녕하세요, 소아청소년과 전문의입니다. 먼저 생선에 대해 말씀드리겠습니다. 부모님들...",안녕하세요 수화 청순과 전문의 가정 됩니다 생선의 먼저 부터 보겠습니다 물어보시는 ...
129,"안녕하세요, 소아청소년과 전문의입니다. 우리 아기가 돌 즈음인데 큰 문제가 있는 것...",안녕하세요 서 총선과 전문이 가정입니다 우리 아기가 돌 7일 않는 데 큰 문제가 있...


In [25]:
df.isnull().sum()

video_url        0
video_id         0
title            0
channel          0
caption          0
clean_caption    0
dtype: int64

# Cleaning Title

In [26]:
df.loc[0,['title']].values

array(['#164  식은땀! 이 정도는 알아두셔야 합니다. : 하정훈의 육아이야기'], dtype=object)

In [ ]:
import re

def cleaning_title(data:str) -> str:
  title = re.sub(r"#\s*\d+", "", data).strip()
  title = title.replace("하정훈", "소아청소년과 전문의")
  
  return title.split(":")[0].strip()

In [37]:
df['title'] = df['title'].map(cleaning_title)

In [38]:
df.loc[0,['title']].values

array(['식은땀! 이 정도는 알아두셔야 합니다.'], dtype=object)

# 학습데이터셋 컬럼 생성 

In [39]:
df['title_caption'] = df.apply(
    lambda row: "[title] " + str(row['title']) 
                + " [caption] " + str(row['clean_caption']), axis=1
)

In [40]:
df[['title', 'clean_caption', 'title_caption']].head(2)

,title,clean_caption,title_caption
0,식은땀! 이 정도는 알아두셔야 합니다.,"안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가...","[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요,..."
2,"식중독, 단순 배탈과 다릅니다. 주의사항 알아봅시다","안녕하세요, 소아청소년과 전문의입니다. 날이 더운 여름에는 식중독이 많이 발생합니다...","[title] 식중독, 단순 배탈과 다릅니다. 주의사항 알아봅시다 [caption]..."


# 필요없는 컬럼 삭제 

In [43]:
try:
    df.drop(columns=["channel", "video_id", "caption"], inplace=True)
except:
    pass 

In [42]:
df.head(2)

,video_url,title,clean_caption,title_caption
0,https://www.youtube.com/watch?v=8GwNlF44ntY&li...,식은땀! 이 정도는 알아두셔야 합니다.,"안녕하세요, 소아청소년과 전문의입니다. 이제는 많이 줄긴 했지만 아직도 우리 아이가...","[title] 식은땀! 이 정도는 알아두셔야 합니다. [caption] 안녕하세요,..."
2,https://www.youtube.com/watch?v=ZO1ffVTWsJc,"식중독, 단순 배탈과 다릅니다. 주의사항 알아봅시다","안녕하세요, 소아청소년과 전문의입니다. 날이 더운 여름에는 식중독이 많이 발생합니다...","[title] 식중독, 단순 배탈과 다릅니다. 주의사항 알아봅시다 [caption]..."


# 학습 데이터셋 저장 

In [44]:
df.shape

(129, 4)

In [45]:
df.to_csv("./data/cleaning/illnesses.csv", index=False, header=True)